In [2]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_squared_error

In [3]:
# CSV'yi oku
df = pd.read_csv("all_data_ratio.csv", encoding="utf-8-sig")

In [4]:
# Tahmin edilecek değişkenler
tahmin_edilecekler = ['uzman', 'pratisyen', 'asistan', 'disci', 'eczaci', 'ebe', 'diger', 'hemsire', 'hastane', 'yatak']
sehirler = df['il_adı'].unique()

In [5]:
# Sonuçları topla
tum_sonuclar = []

In [10]:
for sehir in sehirler:
    df_sehir = df[df['il_adı'] == sehir]

    if df_sehir[df_sehir['yıl'] <= 2021].shape[0] < 5:
        continue

    for hedef in tahmin_edilecekler:
        df_target = df_sehir[['yıl', 'nufus', hedef]].dropna()
        if df_target.empty or df_target[df_target["yıl"] <= 2021].shape[0] == 0:
            continue

        # Train / Val / Test ayır
        train = df_target[df_target['yıl'] <= 2021]
        val = df_target[df_target['yıl'] == 2022]
        test = df_target[df_target['yıl'] == 2023]

        X_train, y_train = train[['yıl', 'nufus']], train[hedef]
        X_val, y_val = val[['yıl', 'nufus']], val[hedef] if not val.empty else (None, None)
        X_test, y_test = test[['yıl', 'nufus']], test[hedef] if not test.empty else (None, None)

        # Linear Regression
        lr_model = LinearRegression()
        lr_model.fit(X_train, y_train)
        y_pred_lr = lr_model.predict(X_test) if not test.empty else [np.nan]

        # Random Forest
        rf_model = RandomForestRegressor(n_estimators=100, random_state=42)
        rf_model.fit(X_train, y_train)
        y_pred_rf = rf_model.predict(X_test) if not test.empty else [np.nan]
        
        tum_sonuclar.append({
            "Şehir": sehir,
            "Hedef": hedef,
            "Tahmin_LR": y_pred_lr[0],
            "Tahmin_RF": y_pred_rf[0]
        })

In [13]:
# Sonuçları dataframe'e çevir
df_sonuclar = pd.DataFrame(tum_sonuclar)

# CSV olarak kaydet
df_sonuclar.to_csv("model_sonuclari.csv", index=False)